# ស្ទូឌីយោ Colab T4 — ញែកភ្លេង + សម្អាតសំឡេង

Notebook នេះបើក service មួយដែលធ្វើការងារពីរ៖

1. **ញែកភ្លេង** — Demucs បំបែក vocals / instrumental (ដូចផ្ទាំងទី១ នៃសាយ)
2. **សម្អាតសំឡេង** — UVR ដកហ៊ឹស/កង្ហារ (DeNoise) និង echo/reverb (DeEcho-DeReverb)

រត់ cell ពីលើចុះក្រោម។ cell ៦ នឹងបោះពុម្ព URL `https://xxxx.trycloudflare.com`
— ចម្លងវាទៅកាត «ញែកភ្លេង · Demucs API» ក្នុងសាយ។

> T4 ឥតគិតថ្លៃ ~១២ ម៉ោង/សប្តាហ៍ ហើយ session ដាច់ពេលអសកម្ម។ URL ថ្មីរាល់ដងដែលបើកឡើងវិញ។

## 1. ពិនិត្យ GPU

បើមិនឃើញ `Tesla T4` → Runtime → Change runtime type → T4 GPU។

In [ ]:
!nvidia-smi -L

## 2. តំឡើង (លើកដំបូង ~៣–៦ នាទី)

In [ ]:
%%bash
set -e
pip install -q --upgrade pip
pip install -q demucs "audio-separator[gpu]" fastapi "uvicorn[standard]" python-multipart
apt-get -qq update && apt-get -qq install -y ffmpeg
echo 'តំឡើងរួចរាល់'


## 3. ម៉ូឌែលសម្អាតសំឡេងមានអ្វីខ្លះ

ឈ្មោះម៉ូឌែលប្តូរតាមកំណែ — មើលបញ្ជីពិតជាមុនសិន។

In [ ]:
!audio-separator --list_models 2>/dev/null | grep -iE 'denoise|reverb|debleed' | head -20 || true

## 4. សរសេរ service

`%%writefile` សរសេរកូដទៅ `/content/service.py` — ដូចគ្នានឹង `tools/colab/service.py` ក្នុង repo។

In [ ]:
%%writefile /content/service.py
"""Colab T4 studio: Demucs separation + UVR vocal cleanup over HTTP.

Open `studio_t4.ipynb` in Colab, run the cells top to bottom, then paste the
printed `https://….trycloudflare.com` address into the website's stem panel.

Endpoints
  GET  /health            what this service is, and which device it runs on
  POST /separate          Demucs two-stem: vocals + instrumental
  POST /enhance           UVR cleanup:  task=denoise | dereverb | debleed
  GET  /files/<job>/<name>  the produced audio, for the site to download

Auth: set `SERVICE_KEY`, then send it as `Authorization: Bearer …`,
`x-api-key: …` or `?key=…`. With no key the service is open — acceptable inside
a private Colab session, never on a public tunnel.
"""

import os
import shlex
import shutil
import subprocess
import sys
import tempfile
import uuid
from pathlib import Path

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import FileResponse

SERVICE_KEY = (os.environ.get("SERVICE_KEY") or "").strip()
DEMUCS_MODEL = (os.environ.get("DEMUCS_MODEL") or "htdemucs").strip()
DEMUCS_CMD = os.environ.get("DEMUCS_CMD") or f"{shlex.quote(sys.executable)} -m demucs"
ENHANCE_CMD = os.environ.get("ENHANCE_CMD") or "audio-separator"

# The UVR models that clean a vocal stem. `audio-separator --list_models` prints
# the exact file names for the installed release; these are the usual ones.
ENHANCE_MODELS = {
    "denoise": os.environ.get("ENHANCE_DENOISE_MODEL", "UVR-DeNoise.pth"),
    "dereverb": os.environ.get("ENHANCE_DEREVERB_MODEL", "UVR-DeEcho-DeReverb.pth"),
    "debleed": os.environ.get(
        "ENHANCE_DEBLEED_MODEL", "mel_band_roformer_denoise_debleed_gabox.ckpt"
    ),
}

# Where the finished stems are written, and what the site downloads.
FILES = Path(os.environ.get("SERVICE_FILES") or "/content/stems")
FILES.mkdir(parents=True, exist_ok=True)

UPLOAD_FIELDS = ("audio", "file", "video", "track", "input", "song")
AUDIO_SUFFIXES = (".wav", ".flac", ".mp3")

app = FastAPI(title="Colab studio service")


# --------------------------------------------------------------------- helpers
def _presented_key(request: Request) -> str:
    auth = request.headers.get("authorization", "")
    bearer = auth[7:].strip() if auth.lower().startswith("bearer ") else ""
    return (
        request.headers.get("x-api-key")
        or bearer
        or request.query_params.get("key")
        or ""
    ).strip()


def _require_key(request: Request) -> None:
    if SERVICE_KEY and _presented_key(request) != SERVICE_KEY:
        raise HTTPException(status_code=401, detail="invalid service key")


async def _save_upload(form, tmpdir: Path) -> Path:
    upload = None
    for name in UPLOAD_FIELDS:
        value = form.get(name)
        if value is not None and getattr(value, "filename", None):
            upload = value
            break
    if upload is None:
        raise HTTPException(status_code=400, detail="no audio file in the request")

    suffix = Path(upload.filename or "input.wav").suffix or ".wav"
    target = tmpdir / f"input{suffix}"
    with open(target, "wb") as handle:
        while True:
            chunk = await upload.read(1024 * 1024)
            if not chunk:
                break
            handle.write(chunk)
    if target.stat().st_size == 0:
        raise HTTPException(status_code=400, detail="the uploaded file was empty")
    return target


def _run(command: list[str], timeout: int = 3600) -> None:
    try:
        subprocess.run(command, check=True, capture_output=True, timeout=timeout)
    except subprocess.CalledProcessError as err:
        detail = (err.stderr or b"").decode("utf-8", "replace")[-600:]
        raise HTTPException(
            status_code=502, detail=f"{command[0]} failed: {detail or err}"
        )
    except subprocess.TimeoutExpired:
        raise HTTPException(status_code=504, detail="the model took too long")


def _device() -> str:
    try:
        import torch  # imported late: it is heavy

        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "unknown"


def _public_url(request: Request, job: str, name: str) -> str:
    base = str(request.base_url).rstrip("/")
    return f"{base}/files/{job}/{name}"


# -------------------------------------------------------------------- endpoints
@app.get("/")
def index():
    return {
        "service": "colab-studio",
        "endpoints": ["/health", "/separate", "/enhance", "/files/{job}/{name}"],
    }


@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": _device(),
        "model": DEMUCS_MODEL,
        "enhance_models": ENHANCE_MODELS,
        "busy": False,
    }


@app.post("/separate")
@app.post("/api/separate")
@app.post("/demucs")
async def separate(request: Request):
    _require_key(request)
    form = await request.form()
    job = uuid.uuid4().hex[:12]
    work = FILES / job
    work.mkdir(parents=True, exist_ok=True)
    tmp = Path(tempfile.mkdtemp())

    try:
        source = await _save_upload(form, tmp)

        # Demucs reads WAV; anything else (MP4, MP3, M4A…) is decoded first, so a
        # video upload works exactly like an audio one.
        # A dedicated name: an uploaded WAV keeps its own `input.wav`, and
        # rewriting that path in place would make ffmpeg refuse to read and write
        # the same file.
        wav = tmp / "demucs_input.wav"
        _run(
            [
                "ffmpeg",
                "-y",
                "-v",
                "error",
                "-i",
                str(source),
                "-ar",
                "44100",
                "-ac",
                "2",
                str(wav),
            ]
        )

        model = str(form.get("model") or DEMUCS_MODEL).strip() or DEMUCS_MODEL
        out = tmp / "out"
        _run(
            shlex.split(DEMUCS_CMD)
            + ["--two-stems", "vocals", "-n", model, "-o", str(out), str(wav)]
        )

        vocals = next(iter(sorted(out.rglob("vocals.*"))), None)
        instrumental = next(iter(sorted(out.rglob("no_vocals.*"))), None)
        if vocals is None or instrumental is None:
            raise HTTPException(status_code=502, detail="Demucs returned no stems")

        shutil.copyfile(vocals, work / f"vocals{vocals.suffix}")
        shutil.copyfile(instrumental, work / f"no_vocals{instrumental.suffix}")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    return {
        "status": "ok",
        "model": model,
        "vocals": _public_url(request, job, f"vocals{vocals.suffix}"),
        "instrumental": _public_url(request, job, f"no_vocals{instrumental.suffix}"),
        "stems": [
            {"name": "vocals", "url": _public_url(request, job, f"vocals{vocals.suffix}")},
            {
                "name": "instrumental",
                "url": _public_url(request, job, f"no_vocals{instrumental.suffix}"),
            },
        ],
    }


@app.post("/enhance")
@app.post("/api/enhance")
@app.post("/clean")
async def enhance(request: Request):
    _require_key(request)
    form = await request.form()
    task = str(form.get("task") or "denoise").strip().lower()
    model = str(form.get("model") or ENHANCE_MODELS.get(task, ENHANCE_MODELS["denoise"])).strip()

    job = uuid.uuid4().hex[:12]
    work = FILES / job
    work.mkdir(parents=True, exist_ok=True)
    tmp = Path(tempfile.mkdtemp())

    try:
        source = await _save_upload(form, tmp)
        _run([ENHANCE_CMD, str(source), "-m", model, "--output_dir", str(work)])
        produced = [p for p in work.iterdir() if p.suffix.lower() in AUDIO_SUFFIXES]
        if not produced:
            raise HTTPException(status_code=502, detail="the model produced no audio")
        # A separator model also writes the part it removed; the cleaned voice is
        # the file whose name says so.
        cleaned = next((p for p in produced if "vocal" in p.name.lower()), produced[0])
        for leftover in produced:
            if leftover != cleaned:
                leftover.unlink(missing_ok=True)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    return {
        "status": "ok",
        "task": task,
        "model": model,
        "clean": _public_url(request, job, cleaned.name),
        "file": _public_url(request, job, cleaned.name),
    }


@app.get("/files/{job}/{name}")
def download(job: str, name: str):
    if "/" in name or ".." in name or "/" in job or ".." in job:
        raise HTTPException(status_code=400, detail="bad path")
    path = FILES / job / name
    if not path.is_file():
        raise HTTPException(status_code=404, detail="not found")
    return FileResponse(path, media_type="audio/wav")


## 5. បើក service

ដាក់ `SERVICE_KEY` ជាពាក្យសម្ងាត់របស់អ្នក (កុំទុកទទេ បើបើកចេញអ៊ីនធឺណិត)។

In [ ]:
import os, time, requests

os.environ['SERVICE_KEY'] = 'change-me-please'
os.environ['DEMUCS_MODEL'] = 'htdemucs'

!pkill -f 'uvicorn service:app' || true
!cd /content && nohup env SERVICE_KEY=$SERVICE_KEY DEMUCS_MODEL=$DEMUCS_MODEL \
    uvicorn service:app --host 0.0.0.0 --port 8920 > /content/service.log 2>&1 &

time.sleep(6)
print(requests.get('http://127.0.0.1:8920/health', timeout=20).json())


## 6. បើក tunnel យក URL

ចម្លopy URL ខាងក្រោមទៅកាត «ញែកភ្លេង · Demucs API» ក្នុងសាយ។

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!pkill -f cloudflared || true
!nohup cloudflared tunnel --url http://localhost:8920 > /content/tunnel.log 2>&1 &

import re, time
time.sleep(10)
log = open('/content/tunnel.log').read()
match = re.search(r'https://[-\w]+\.trycloudflare\.com', log)
print('URL :', match.group(0) if match else 'រកមិនឃើញ — មើល /content/tunnel.log')
print('KEY :', os.environ.get('SERVICE_KEY', ''))


## 7. សាកល្បងញែកភ្លេង

ដាក់បទទៅ `/content/in/song.mp3` សិន។

In [ ]:
!mkdir -p /content/in
!cd /content && curl -sS -H "Authorization: Bearer $SERVICE_KEY" \
    -F "audio=@/content/in/song.mp3" http://127.0.0.1:8920/separate | head -c 600


## 8. សម្អាត vocal stem (UVR)

```
task=denoise   → UVR-DeNoise.pth          (ហ៊ឹស, កង្ហារ, ខ្យល់)
task=dereverb  → UVR-DeEcho-DeReverb.pth  (echo, សំឡេងរង្គោះរង្គើ)
task=debleed   → RoFormer denoise/debleed (ភ្លេងលូកចូល)
```

ដាក់ path របស់ `vocals.wav` ពី cell ៧ ជំនួស `<job>`។

In [ ]:
!cd /content && curl -sS -H "Authorization: Bearer $SERVICE_KEY" \
    -F "audio=@/content/stems/<job>/vocals.wav" -F "task=denoise" \
    http://127.0.0.1:8920/enhance | head -c 400


## ចំណាំ

* Job នៅសាយឈប់នៅជំហានញែក → tunnel ឬ session បានបិទ; បើក cell ៥–៦ ម្តងទៀត ហើយ paste URL ថ្មី។
* `/enhance` មិនទាន់ហៅដោយសាយ — ត្រៀមទុកសម្រាប់ជំហាន «សម្អាតជ្រៅ» (សាកដោយ curl បាន)។
* Stem រក្សាក្នុង `/content/stems/<job>/` រហូត session ដាច់។
* កុំបើក `SERVICE_KEY` ទទេលើ tunnel សាធារណៈ — អ្នកណាមាន URL នឹងប្រើ GPU របស់អ្នកបាន។